In [3]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import json

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

Log loaded. Rows: 48
PROCESSED_DIR: C:\Users\mjbou\governance-framework\data\processed


In [5]:
sources = [
    {
        "source_id": "VDEM",
        "source_name": "Varieties of Democracy (V-Dem)",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv on full dataset download",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "1789-present",
        "highest_tier": "P1",
        "notes": "Single largest source in framework. ~4000 variables. Download full dataset once; filter to needed variables. Codebook study essential before metric selection."
    },
    {
        "source_id": "WGI",
        "source_name": "World Bank Worldwide Governance Indicators",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "annual",
        "coverage_countries": 215,
        "coverage_years": "1996-present",
        "highest_tier": "P1",
        "notes": "Used as concept primary in 3 concepts (GE, PS, RQ); category cross-check elsewhere. wbgapi is clean and well-documented."
    },
    {
        "source_id": "WDI",
        "source_name": "World Bank World Development Indicators",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "annual",
        "coverage_countries": 217,
        "coverage_years": "1960-present",
        "highest_tier": "P1",
        "notes": "Used for service delivery sector indicators. Select specific indicators only — dataset is very broad."
    },
    {
        "source_id": "WJP",
        "source_name": "World Justice Project Rule of Law Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 142,
        "coverage_years": "2012-present",
        "highest_tier": "P1",
        "notes": "Borderline coverage (~142 countries). Factors allocated by concept: F2=Corruption, F3=Legal Quality+Transparency, F4=Legal Quality, F5=Personal Security+Stability, F6=Regulatory Quality, F7+F8=Judicial Independence. Download Excel from worldjusticeproject.org."
    },
    {
        "source_id": "FH_FIW",
        "source_name": "Freedom House Freedom in the World",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 210,
        "coverage_years": "1973-present",
        "highest_tier": "P1",
        "notes": "Disciplined sub-component extraction required. Sub-categories by concept: A=Electoral Process, D=Expression+Belief (Civil Liberties+Media), E=Associational Rights (Civil Society), G=Personal Autonomy (Civil Liberties). Do NOT use composite FIW score."
    },
    {
        "source_id": "FSI",
        "source_name": "Fragile States Index (Fund for Peace)",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 179,
        "coverage_years": "2006-present",
        "highest_tier": "P1",
        "notes": "Different indicators by concept: P1 (Factionalized Elites) + S1 (Group Grievance) in Political Settlement; P2 (Public Services) in Service Delivery; C1 (Security Apparatus) in State Capacity."
    },
]

# Preview
pd.DataFrame(sources)

,source_id,source_name,access_method,python_approach,update_frequency,coverage_countries,coverage_years,highest_tier,notes
0,VDEM,Varieties of Democracy (V-Dem),bulk_download,pd.read_csv on full dataset download,annual,180,1789-present,P1,Single largest source in framework. ~4000 vari...
1,WGI,World Bank Worldwide Governance Indicators,api,wbgapi,annual,215,1996-present,P1,"Used as concept primary in 3 concepts (GE, PS,..."
2,WDI,World Bank World Development Indicators,api,wbgapi,annual,217,1960-present,P1,Used for service delivery sector indicators. S...
3,WJP,World Justice Project Rule of Law Index,bulk_download,pd.read_excel,annual,142,2012-present,P1,Borderline coverage (~142 countries). Factors ...
4,FH_FIW,Freedom House Freedom in the World,bulk_download,pd.read_excel,annual,210,1973-present,P1,Disciplined sub-component extraction required....
5,FSI,Fragile States Index (Fund for Peace),bulk_download,pd.read_excel,annual,179,2006-present,P1,Different indicators by concept: P1 (Factional...


In [3]:
sources += [
    {
        "source_id": "IMF_FISCAL_RULES",
        "source_name": "IMF Fiscal Rules Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 100,
        "coverage_years": "1985-present",
        "highest_tier": "P1",
        "notes": "Used in Macroeconomic policy framework. Covers existence, design, and compliance of fiscal rules. Download from IMF website."
    },
    {
        "source_id": "IMF_AREAER",
        "source_name": "IMF Annual Report on Exchange Arrangements and Exchange Restrictions",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 190,
        "coverage_years": "1950-present",
        "highest_tier": "P1",
        "notes": "De jure exchange rate regime classification. Used in Macroeconomic policy framework alongside Reinhart-Rogoff de facto classifications."
    },
    {
        "source_id": "IMF_IMAPP",
        "source_name": "IMF Integrated Macroprudential Policy Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 130,
        "coverage_years": "1990-present",
        "highest_tier": "P1",
        "notes": "Macroprudential policy adoption. Used in Macroeconomic policy framework. Download from IMF website."
    },
    {
        "source_id": "IMF_SPI",
        "source_name": "World Bank Statistical Performance Indicators",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "annual",
        "coverage_countries": 174,
        "coverage_years": "2016-present",
        "highest_tier": "P1",
        "notes": "Primary source for Statistical and informational infrastructure. Covers data infrastructure, sources, products, services, use."
    },
    {
        "source_id": "ROMELLI_CBI",
        "source_name": "Romelli Central Bank Independence Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 155,
        "coverage_years": "1923-2021",
        "highest_tier": "P1",
        "notes": "Current state-of-the-art for CBI. Updates require new paper/replication release — not annually updated. Verify currency at metric pass."
    },
    {
        "source_id": "DINCER_CB",
        "source_name": "Dincer-Eichengreen Central Bank Transparency Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 120,
        "coverage_years": "1998-present",
        "highest_tier": "P1",
        "notes": "CB transparency dimension for Macroeconomic policy framework. Update status and most recent year to verify at metric pass."
    },
    {
        "source_id": "REINHART_ROGOFF",
        "source_name": "Reinhart-Rogoff Exchange Rate Classifications",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 190,
        "coverage_years": "1940-present",
        "highest_tier": "P1",
        "notes": "De facto exchange rate regime. Complements AREAER de jure. Update frequency irregular — verify most recent release."
    },
    {
        "source_id": "UNODC_HOMICIDE",
        "source_name": "UNODC Homicide Statistics",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "1990-present",
        "highest_tier": "P1",
        "notes": "Gold standard for Personal security and order. High S/N. Download from UNODC data portal."
    },
    {
        "source_id": "PTS",
        "source_name": "Political Terror Scale",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 190,
        "coverage_years": "1976-present",
        "highest_tier": "P1",
        "notes": "Used in both Personal security (P1) and Civil liberties (P2). Indicator repetition tracked. Download from politicalterrorscale.org."
    },
    {
        "source_id": "POWELL_THYNE",
        "source_name": "Powell-Thyne Coup Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "1950-present",
        "highest_tier": "P1",
        "notes": "Event-level coup data. Used in Political stability (P1) and Political settlement (Sp). Requires country-year panel construction from event data."
    },
    {
        "source_id": "UCDP",
        "source_name": "Uppsala Conflict Data Program",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "1946-present",
        "highest_tier": "P1",
        "notes": "Gold standard for armed conflict. Used in Political stability. Multiple datasets (GED, PRIO-Grid, dyadic). Use GED for country-year panel."
    },
    {
        "source_id": "ACLED",
        "source_name": "Armed Conflict Location and Event Data",
        "access_method": "api",
        "python_approach": "requests (ACLED API requires registration)",
        "update_frequency": "continuous",
        "coverage_countries": 250,
        "coverage_years": "1997-present",
        "highest_tier": "P1",
        "notes": "Real-time event data. API requires free registration and key. Complements UCDP with broader event types."
    },
    {
        "source_id": "TI_CPI",
        "source_name": "Transparency International Corruption Perceptions Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "1995-present",
        "highest_tier": "P1",
        "notes": "Standard corruption measure. Used in Control of corruption (P1). Aggregator of 13 underlying sources."
    },
    {
        "source_id": "RSF_WPFI",
        "source_name": "RSF World Press Freedom Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "2002-present",
        "highest_tier": "P1",
        "notes": "Primary for Media freedom. Download from RSF website. Methodology changed significantly in 2023 — treat pre/post 2023 as partially discontinuous series."
    },
    {
        "source_id": "CPJ",
        "source_name": "Committee to Protect Journalists",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv or requests",
        "update_frequency": "continuous",
        "coverage_countries": 200,
        "coverage_years": "1992-present",
        "highest_tier": "P1",
        "notes": "Journalist safety outcome measure for Media freedom. Event-level data requiring country-year aggregation. CPJ database accessible via website download."
    },
    {
        "source_id": "CIVICUS",
        "source_name": "CIVICUS Monitor",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel or requests",
        "update_frequency": "annual",
        "coverage_countries": 197,
        "coverage_years": "2017-present",
        "highest_tier": "P1",
        "notes": "Used in Political participation (P1) and Civil society space (P1). Categorical scoring (Open/Narrowed/Obstructed/Repressed/Closed). Short time series."
    },
    {
        "source_id": "IDEA_EMB",
        "source_name": "IDEA Electoral Management Design Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 220,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "EMB design and independence for Electoral process. Structural/design data rather than time series. Update frequency to verify."
    },
    {
        "source_id": "PEI",
        "source_name": "Electoral Integrity Project — Perceptions of Electoral Integrity",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "per_election",
        "coverage_countries": 170,
        "coverage_years": "2012-present",
        "highest_tier": "P1",
        "notes": "Per-election cadence requires constructing most-recent-election panel. 49 indicators across 11 dimensions. Download from Electoral Integrity Project website."
    },
    {
        "source_id": "CCP",
        "source_name": "Comparative Constitutions Project",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 200,
        "coverage_years": "1789-present",
        "highest_tier": "P1",
        "notes": "De jure constitutional framework. Used across Legal quality, Judicial independence, Property rights, Legislative checks, Electoral process. Constitute Project is the searchable interface."
    },
    {
        "source_id": "DPI",
        "source_name": "Database of Political Institutions",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "1975-present",
        "highest_tier": "P2",
        "notes": "Party fragmentation and government composition proxies. Used in Political settlement (P2). World Bank hosted."
    },
    {
        "source_id": "GPI",
        "source_name": "Global Peace Index (IEP)",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 163,
        "coverage_years": "2008-present",
        "highest_tier": "P2",
        "notes": "Used in Political stability (P2) and Personal security (P2). Pre-aggregated composite — use domain-level scores not headline."
    },
    {
        "source_id": "ODIN",
        "source_name": "Open Data Inventory (Open Data Watch)",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "biennial",
        "coverage_countries": 195,
        "coverage_years": "2016-present",
        "highest_tier": "P1",
        "notes": "Best-in-class for accessibility/openness sub-dimension of Statistical infrastructure. Also supplementary in Government transparency."
    },
    {
        "source_id": "PEFA",
        "source_name": "Public Expenditure and Financial Accountability",
        "access_method": "tier3_pdf",
        "python_approach": "PDF extraction — pdfplumber + manual review",
        "update_frequency": "per_country_4_7yr",
        "coverage_countries": 150,
        "coverage_years": "2001-present",
        "highest_tier": "P1",
        "notes": "Gold standard for PFM. Irregular per-country timing is the key operational challenge. PEFA Secretariat portal has structured data for some indicators. Verify portal coverage before committing to PDF extraction."
    },
    {
        "source_id": "OBS",
        "source_name": "Open Budget Survey (IBP)",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "biennial",
        "coverage_countries": 120,
        "coverage_years": "2006-present",
        "highest_tier": "P1",
        "notes": "Used in PFM (P1) and Government transparency (P2). Biennial cadence. Download from IBP website."
    },
    {
        "source_id": "FATF",
        "source_name": "FATF Mutual Evaluation Ratings",
        "access_method": "tier2_structured",
        "python_approach": "requests or pd.read_html from fatf-gafi.org",
        "update_frequency": "per_country_10yr",
        "coverage_countries": 200,
        "coverage_years": "2004-present",
        "highest_tier": "P1",
        "notes": "AML/CFT compliance ratings. Structured ratings on fatf-gafi.org are scrapeable. Full reports are PDFs (Tier 3). Per-country cycle ~10 years with intermediate follow-ups."
    },
    {
        "source_id": "BASEL_AML",
        "source_name": "Basel AML Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 205,
        "coverage_years": "2012-present",
        "highest_tier": "P1",
        "notes": "AML/CFT risk composite synthesising FATF and other sources. Annual, free, broad coverage. Practical workhorse for financial sector regulatory concept."
    },
    {
        "source_id": "HERITAGE_TR",
        "source_name": "Heritage Index of Economic Freedom — Trade Freedom",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "1995-present",
        "highest_tier": "P1",
        "notes": "Used in Trade governance. Lower ideological loading for trade openness dimension than other Heritage components."
    },
    {
        "source_id": "HERITAGE_PR",
        "source_name": "Heritage Index of Economic Freedom — Property Rights",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "1995-present",
        "highest_tier": "P1",
        "notes": "Used in Property rights and contract enforcement. Lower loading than other Heritage components for this dimension."
    },
    {
        "source_id": "WB_LPI",
        "source_name": "World Bank Logistics Performance Index",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "irregular",
        "coverage_countries": 139,
        "coverage_years": "2007-present",
        "highest_tier": "P1",
        "notes": "Trade administration quality. 5-year update gap historically. Verify current cadence. Available via WB API."
    },
    {
        "source_id": "OECD_TFI",
        "source_name": "OECD Trade Facilitation Indicators",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "biennial",
        "coverage_countries": 163,
        "coverage_years": "2012-present",
        "highest_tier": "P1",
        "notes": "11 indicators covering trade administration. Updated every 2-3 years. Download from OECD website."
    },
    {
        "source_id": "KOF_TRADE",
        "source_name": "KOF Globalisation Index — Trade subindex",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "1970-present",
        "highest_tier": "P1",
        "notes": "De jure and de facto trade openness. Lower ideological framing than Heritage/Fraser. Download from KOF Swiss Economic Institute."
    },
    {
        "source_id": "UNCTAD_NTM",
        "source_name": "UNCTAD Non-Tariff Measures Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 110,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Only good cross-country NTM source. Borderline coverage (~110). Update frequency irregular. Download from UNCTAD website."
    },
    {
        "source_id": "YALE_EPI",
        "source_name": "Yale Environmental Performance Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "biennial",
        "coverage_countries": 180,
        "coverage_years": "2006-present",
        "highest_tier": "P1",
        "notes": "Used in Environmental governance. Use policy/institutional sub-components only — not headline composite. Biennial."
    },
    {
        "source_id": "CLIMATE_LAWS",
        "source_name": "LSE Grantham Climate Laws Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv or requests",
        "update_frequency": "continuous",
        "coverage_countries": 200,
        "coverage_years": "1800-present",
        "highest_tier": "P1",
        "notes": "De jure environmental and climate framework. Continuously updated. Download from climatecasechart.com / climate-laws.org."
    },
    {
        "source_id": "ND_GAIN",
        "source_name": "ND-GAIN Country Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 192,
        "coverage_years": "1995-present",
        "highest_tier": "P1",
        "notes": "Governance and readiness sub-scores for Environmental governance. Use sub-scores not headline index. Download from gain.nd.edu."
    },
    {
        "source_id": "IRENA_CAPACITY",
        "source_name": "IRENA Renewables Capacity Statistics",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "2000-present",
        "highest_tier": "P1",
        "notes": "Renewables outcomes for Environmental governance. High S/N for renewables specifically. Download from IRENA website."
    },
    {
        "source_id": "IRENA_POLICY",
        "source_name": "IRENA Renewable Energy Policies Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel or requests",
        "update_frequency": "continuous",
        "coverage_countries": 200,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Renewables policy adoption. Complements IRENA capacity outcomes. Download from IRENA website."
    },
    {
        "source_id": "WB_CARBON",
        "source_name": "World Bank Carbon Pricing Dashboard",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel or requests",
        "update_frequency": "annual",
        "coverage_countries": 100,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Carbon pricing existence and design. Universal coverage for countries with carbon pricing. Download from World Bank."
    },
    {
        "source_id": "HANSON_SIGMAN",
        "source_name": "Hanson-Sigman State Capacity Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 169,
        "coverage_years": "1960-2021",
        "highest_tier": "Sp",
        "notes": "Supplementary cross-check for State capacity. Last update 2021. Double-counting caveat — incorporates V-Dem and other framework sources."
    },
    {
        "source_id": "WTO_TFA",
        "source_name": "WTO Trade Facilitation Agreement Implementation",
        "access_method": "bulk_download",
        "python_approach": "requests or pd.read_html",
        "update_frequency": "continuous",
        "coverage_countries": 164,
        "coverage_years": "2017-present",
        "highest_tier": "P1",
        "notes": "Country commitments and implementation. All WTO members. Continuously updated on WTO website."
    },
    {
        "source_id": "IPU_PARLINE",
        "source_name": "IPU Parline Database",
        "access_method": "bulk_download",
        "python_approach": "requests or pd.read_html",
        "update_frequency": "continuous",
        "coverage_countries": 190,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Legislative structural features and oversight powers. Universal coverage. Authoritative source. Continuously updated."
    },
    {
        "source_id": "RTI_RATING",
        "source_name": "Centre for Law and Democracy / Access Info Europe RTI Rating",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel or requests",
        "update_frequency": "irregular",
        "coverage_countries": 138,
        "coverage_years": "2011-present",
        "highest_tier": "P2",
        "notes": "FOI/RTI legislation quality. Borderline coverage (~138). Used in Media freedom (P2) and Government transparency (P1)."
    },
    {
        "source_id": "TI_POLFINANCE",
        "source_name": "Transparency International Political Finance Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 180,
        "coverage_years": "varies",
        "highest_tier": "P2",
        "notes": "Unique to Government transparency concept. Political party and campaign finance transparency."
    },
    {
        "source_id": "WIPO",
        "source_name": "WIPO IP Statistics",
        "access_method": "api",
        "python_approach": "requests (WIPO API)",
        "update_frequency": "annual",
        "coverage_countries": 190,
        "coverage_years": "varies",
        "highest_tier": "P2",
        "notes": "IP protection dimension for Property rights. Download from WIPO IP Statistics portal."
    },
    {
        "source_id": "ILO_SOCIAL",
        "source_name": "ILO Social Security Coverage",
        "access_method": "api",
        "python_approach": "requests (ILO API) or pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 150,
        "coverage_years": "varies",
        "highest_tier": "P2",
        "notes": "Formality via state administrative systems. Proxy for state reach for State capacity concept."
    },
    {
        "source_id": "WB_INFORMAL",
        "source_name": "World Bank Informal Economy Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 160,
        "coverage_years": "varies",
        "highest_tier": "P2",
        "notes": "Informality as proxy for state reach. Used in State capacity (P2). Update frequency to verify."
    },
    {
        "source_id": "FRASER_REG",
        "source_name": "Fraser Economic Freedom — Regulation area",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 165,
        "coverage_years": "1970-present",
        "highest_tier": "P2",
        "notes": "Used in Regulatory quality (P2) with framing caveats. Download from Fraser Institute."
    },
    {
        "source_id": "FRASER_LEGAL",
        "source_name": "Fraser Economic Freedom — Legal System and Property Rights",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 165,
        "coverage_years": "1970-present",
        "highest_tier": "P2",
        "notes": "Used selectively in Property rights (P2) — property sub-components only. Judicial independence content stays in dedicated concept."
    },
    {
        "source_id": "PEW_GRI",
        "source_name": "Pew Government Restrictions Index and Social Hostilities Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "2007-present",
        "highest_tier": "P2",
        "notes": "Religious freedom dimension for Civil liberties. Tier 2 — religious freedom less central to political accountability than expression/dissent."
    },
    {
        "source_id": "WB_WBL",
        "source_name": "World Bank Women, Business and the Law",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "annual",
        "coverage_countries": 190,
        "coverage_years": "1971-present",
        "highest_tier": "P2",
        "notes": "Gender equality dimension for Civil liberties. Direct legal protections measurement. High S/N."
    },
    {
        "source_id": "NELDA",
        "source_name": "National Elections Across Democracy and Autocracy",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 200,
        "coverage_years": "1945-present",
        "highest_tier": "P2",
        "notes": "Event-level election data for Electoral process (P2). Requires country-year construction. Update status to verify."
    },
    {
        "source_id": "IDEA_PARTIP",
        "source_name": "IDEA Global State of Democracy — Participatory Engagement",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 173,
        "coverage_years": "1975-present",
        "highest_tier": "P2",
        "notes": "Used in Political participation (P2). Some V-Dem double-counting given underlying sources. Download from IDEA website."
    },
    {
        "source_id": "BCI",
        "source_name": "Bayesian Corruption Indicator",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 190,
        "coverage_years": "varies",
        "highest_tier": "P2",
        "notes": "Latent-variable cross-check for Control of corruption. Update currency to verify at metric pass."
    },
    {
        "source_id": "GLOBAL_DATA_BAROMETER",
        "source_name": "Global Data Barometer",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 109,
        "coverage_years": "2021-present",
        "highest_tier": "Sp",
        "notes": "Open data dimension. Borderline coverage (~109). Unique to Government transparency. Successor to defunct Open Data Barometer."
    },
    {
        "source_id": "IMF_SPI_SDDS",
        "source_name": "IMF Data Standards Subscriptions (SDDS/eGDDS)",
        "access_method": "bulk_download",
        "python_approach": "requests or pd.read_html",
        "update_frequency": "continuous",
        "coverage_countries": 190,
        "coverage_years": "1996-present",
        "highest_tier": "P2",
        "notes": "De jure standards compliance for Statistical infrastructure. Universal IMF members. Tiered by income group."
    },
    {
        "source_id": "WHO_GHO",
        "source_name": "WHO Global Health Observatory",
        "access_method": "api",
        "python_approach": "requests (WHO GHO API)",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Health-specific direct outputs for Service delivery. Universal coverage. Well-documented API."
    },
    {
        "source_id": "UNESCO_UIS",
        "source_name": "UNESCO Institute for Statistics",
        "access_method": "api",
        "python_approach": "requests (UIS API) or bulk download",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Education-specific direct outputs for Service delivery. Universal coverage. UIS bulk data download also available."
    },
    {
        "source_id": "UNDP_HDI",
        "source_name": "UNDP Human Development Index sub-indicators",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 193,
        "coverage_years": "1990-present",
        "highest_tier": "P1",
        "notes": "Use sub-indicators only (life expectancy, schooling years) — NOT composite HDI. Download from UNDP HDR website."
    },
    {
        "source_id": "WB_HCI",
        "source_name": "World Bank Human Capital Index",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "biennial",
        "coverage_countries": 170,
        "coverage_years": "2018-present",
        "highest_tier": "P2",
        "notes": "Composite cross-check for Service delivery. Short time series. Use as summary cross-check not primary."
    },
    {
        "source_id": "POLITY5",
        "source_name": "Polity5",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 167,
        "coverage_years": "1800-2018",
        "highest_tier": "P2",
        "notes": "Limited use given V-Dem supersession. Durable (Sp) in Political stability; XCONST (P2) in Legislative checks; electoral components (Sp) in Electoral process. Update reliability concern — verify currency."
    },
    {
        "source_id": "LINZER_STATON",
        "source_name": "Linzer-Staton Judicial Independence Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 200,
        "coverage_years": "1948-present",
        "highest_tier": "Sp",
        "notes": "Supplementary cross-check for Judicial independence. Methodologically sophisticated latent variable. Update frequency to verify."
    },
    {
        "source_id": "ICNL",
        "source_name": "ICNL Civic Freedom Monitor",
        "access_method": "tier3_web",
        "python_approach": "requests / manual review",
        "update_frequency": "irregular",
        "coverage_countries": 100,
        "coverage_years": "varies",
        "highest_tier": "Sp",
        "notes": "De jure legal framework for Civil society space. Uneven coverage. Supplementary only."
    },
]

# Full registry
registry_df = pd.DataFrame(sources)
print(f"Total sources: {len(registry_df)}")
# registry_df
# with pd.option_context('display.max_rows', None):
#     display(registry_df)
registry_df

Total sources: 68


,source_id,source_name,access_method,python_approach,update_frequency,coverage_countries,coverage_years,highest_tier,notes
0,VDEM,Varieties of Democracy (V-Dem),bulk_download,pd.read_csv on full dataset download,annual,180,1789-present,P1,Single largest source in framework. ~4000 vari...
1,WGI,World Bank Worldwide Governance Indicators,api,wbgapi,annual,215,1996-present,P1,"Used as concept primary in 3 concepts (GE, PS,..."
2,WDI,World Bank World Development Indicators,api,wbgapi,annual,217,1960-present,P1,Used for service delivery sector indicators. S...
3,WJP,World Justice Project Rule of Law Index,bulk_download,pd.read_excel,annual,142,2012-present,P1,Borderline coverage (~142 countries). Factors ...
4,FH_FIW,Freedom House Freedom in the World,bulk_download,pd.read_excel,annual,210,1973-present,P1,Disciplined sub-component extraction required....
...,...,...,...,...,...,...,...,...,...
63,UNDP_HDI,UNDP Human Development Index sub-indicators,bulk_download,pd.read_csv,annual,193,1990-present,P1,"Use sub-indicators only (life expectancy, scho..."
64,WB_HCI,World Bank Human Capital Index,api,wbgapi,biennial,170,2018-present,P2,Composite cross-check for Service delivery. Sh...
65,POLITY5,Polity5,bulk_download,pd.read_excel,irregular,167,1800-2018,P2,Limited use given V-Dem supersession. Durable ...
66,LINZER_STATON,Linzer-Staton Judicial Independence Index,bulk_download,pd.read_csv,irregular,200,1948-present,Sp,Supplementary cross-check for Judicial indepen...


In [4]:
import os

output_path = os.path.join(PROCESSED_DIR, "source_registry.csv")
registry_df.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {registry_df.shape}")

Written: /Users/boulanger/Documents/governance-framework/data/processed/source_registry.csv
Shape: (68, 9)


In [5]:
# Update WHO_GHO entry to note it's subsumed by WDI
registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'notes'] = (
    "Health workforce and outcomes data sourced from WHO Global Health Workforce Statistics. "
    "All required indicators available via WDI (wbgapi). "
    "Standalone WHO GHO pipeline not built — GHO OData API deprecated end-2025. "
    "Indicators covered: physicians/1000, nurses/1000, hospital beds/1000, UHC coverage index."
)
registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'python_approach'] = 'wbgapi — see WDI pipeline'

# Save updated registry
output_path = os.path.join(PROCESSED_DIR, "source_registry.csv")
registry_df.to_csv(output_path, index=False)
print("Registry updated")
registry_df[registry_df['source_id'] == 'WHO_GHO'][['source_id', 'access_method', 'python_approach', 'notes']]

Registry updated


,source_id,access_method,python_approach,notes
61,WHO_GHO,via_wdi,wbgapi — see WDI pipeline,Health workforce and outcomes data sourced fro...


In [6]:
# Update WHO_GHO entry to reflect it's subsumed by WDI
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'notes'] = (
    "Health workforce and outcomes data sourced from WHO Global Health Workforce Statistics. "
    "All required indicators available via WDI (wbgapi). "
    "Standalone WHO GHO pipeline not built — GHO OData API deprecated end-2025. "
    "Indicators covered: physicians/1000, nurses/1000, hospital beds/1000, UHC coverage index."
)
registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'python_approach'] = 'wbgapi — see WDI pipeline'

# Save
registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'WHO_GHO'][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
   source_id access_method            python_approach
61   WHO_GHO       via_wdi  wbgapi — see WDI pipeline


In [7]:
import json
from pathlib import Path

notebooks_dir = Path(PROJECT_ROOT) / 'notebooks' / 'exploration'
for nb_path in sorted(notebooks_dir.glob('*.ipynb')):
    with open(nb_path) as f:
        nb = json.load(f)
    first_cell = ''.join(nb['cells'][0]['source'])
    if 'PROCESSED_DIR' in first_cell and 'from config import' not in first_cell:
        print(f"⚠️  HARDCODED: {nb_path.name}")
    else:
        print(f"OK: {nb_path.name}")

OK: 01_pdf_extraction.ipynb
OK: 02_source_registry.ipynb
OK: 03_vdem_pipeline.ipynb
OK: 04_wgi_pipeline.ipynb
OK: 05_wjp_pipeline.ipynb
OK: 06_fh_fiw_pipeline.ipynb
OK: 07_fsi_pipeline.ipynb
OK: 08_ti_cpi_pipeline.ipynb
OK: 09_wdi_pipeline.ipynb
OK: 10_imf_spi_pipeline.ipynb
OK: 11_acled_pipeline.ipynb
OK: 12_ucdp_pipeline.ipynb
OK: 13_fraser_pipeline.ipynb
OK: 14_qog_pipeline.ipynb
OK: 15_powell_thyne_pipeline..ipynb


In [8]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'UNESCO_UIS', 'notes'] = (
    "Education indicators sourced from UNESCO UIS, distributed via WDI (wbgapi). "
    "Standalone UNESCO UIS pipeline not built. "
    "Indicators covered: education expenditure % GDP, education expenditure % govt, "
    "pupil-teacher ratios primary and secondary."
)
registry_df.loc[registry_df['source_id'] == 'UNESCO_UIS', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'UNESCO_UIS', 'python_approach'] = 'wbgapi — see WDI pipeline'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'UNESCO_UIS'][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
     source_id access_method            python_approach
62  UNESCO_UIS       via_wdi  wbgapi — see WDI pipeline


In [9]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

# WB_LPI
registry_df.loc[registry_df['source_id'] == 'WB_LPI', 'notes'] = (
    "Logistics Performance Index overall score. Available via WDI (LP.LPI.OVRL.XQ). "
    "Added to WDI pipeline. No standalone LPI pipeline needed."
)
registry_df.loc[registry_df['source_id'] == 'WB_LPI', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'WB_LPI', 'python_approach'] = 'wbgapi — see WDI pipeline'

# WB_HCI
registry_df.loc[registry_df['source_id'] == 'WB_HCI', 'notes'] = (
    "Standard HCI (HD.HCI.OVRL) not available via WB API. "
    "Using HCI+ overall total (HD_HCIP_OVRL_TO) as substitute — same concept, expanded methodology. "
    "Added to WDI pipeline. No standalone HCI pipeline needed."
)
registry_df.loc[registry_df['source_id'] == 'WB_HCI', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'WB_HCI', 'python_approach'] = 'wbgapi — see WDI pipeline'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'].isin(['WB_LPI', 'WB_HCI'])][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
   source_id access_method            python_approach
34    WB_LPI       via_wdi  wbgapi — see WDI pipeline
64    WB_HCI       via_wdi  wbgapi — see WDI pipeline


In [10]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

# WIPO
registry_df.loc[registry_df['source_id'] == 'WIPO', 'notes'] = (
    "Patent and trademark application data available via WDI (wbgapi). "
    "Indicators: IP.PAT.RESD, IP.PAT.NRES, IP.TMK.RSCT, IP.TMK.NRCT. "
    "All added to WDI pipeline. No standalone WIPO pipeline needed."
)
registry_df.loc[registry_df['source_id'] == 'WIPO', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'WIPO', 'python_approach'] = 'wbgapi — see WDI pipeline'

# ILO_SOCIAL
registry_df.loc[registry_df['source_id'] == 'ILO_SOCIAL', 'notes'] = (
    "Social protection coverage indicators available via World Bank API (wbgapi). "
    "Indicators: per_allsp.cov_pop_tot, per_sa_allsa.cov_pop_tot, per_si_allsi.cov_pop_tot. "
    "All added to WDI pipeline. No standalone ILO pipeline needed."
)
registry_df.loc[registry_df['source_id'] == 'ILO_SOCIAL', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'ILO_SOCIAL', 'python_approach'] = 'wbgapi — see WDI pipeline'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'].isin(['WIPO', 'ILO_SOCIAL'])][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
     source_id access_method            python_approach
49        WIPO       via_wdi  wbgapi — see WDI pipeline
50  ILO_SOCIAL       via_wdi  wbgapi — see WDI pipeline


In [11]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'UNDP_HDI', 'notes'] = (
    "HDI sub-indicators available via WDI (wbgapi). "
    "Indicators added to WDI pipeline: SP.DYN.LE00.IN (life expectancy), NY.GNP.PCAP.PP.CD (GNI per capita PPP). "
    "Years of schooling not available via WDI — enrollment rates used as substitute. "
    "Composite HDI index not used per framework design. No standalone UNDP pipeline needed."
)
registry_df.loc[registry_df['source_id'] == 'UNDP_HDI', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'UNDP_HDI', 'python_approach'] = 'wbgapi — see WDI pipeline'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'UNDP_HDI'][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
   source_id access_method            python_approach
63  UNDP_HDI       via_wdi  wbgapi — see WDI pipeline


In [12]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'RSF_WPFI', 'notes'] = (
    "Media freedom concept covered comprehensively by V-Dem primary indicators. "
    "RSF WPFI kept as optional annual manual cross-check only. "
    "No automated pipeline built. "
    "2022+ data: manual download from rsf.org/en/index — new methodology, not comparable to pre-2022. "
    "Pre-2022 data available via OWID but uses old methodology (0=free, 100=unfree) — do not stitch with post-2022. "
    "If manual download needed: go to rsf.org/en/index, download Excel, place in data/raw/."
)
registry_df.loc[registry_df['source_id'] == 'RSF_WPFI', 'access_method'] = 'manual_optional'
registry_df.loc[registry_df['source_id'] == 'RSF_WPFI', 'python_approach'] = 'manual download only — see notes'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'RSF_WPFI'][['source_id', 'access_method']].to_string())

Registry updated
   source_id    access_method
19  RSF_WPFI  manual_optional


In [13]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'GPI', 'notes'] = (
    "Political stability concept well covered by UCDP, FSI, WGI PV and V-Dem. "
    "GPI adds militarization and societal safety dimensions as secondary cross-check. "
    "No automated pipeline — IEP blocks direct downloads. "
    "Manual download: go to visionofhumanity.org/maps/, click Download Data, save Excel to data/raw/. "
    "Historical panel available from 2008. Annual release in June."
)
registry_df.loc[registry_df['source_id'] == 'GPI', 'access_method'] = 'manual_annual'
registry_df.loc[registry_df['source_id'] == 'GPI', 'python_approach'] = 'manual download — see notes'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'GPI'][['source_id', 'access_method']].to_string())

Registry updated
   source_id  access_method
26       GPI  manual_annual


In [14]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

# FRASER_REG
registry_df.loc[registry_df['source_id'] == 'FRASER_REG', 'notes'] = (
    "Fraser EFW Area 5 (Regulation). Automated pipeline — scrapes annual report page to find XLSX URL. "
    "Uses EFW Panel Dataset sheet (chain-linked, base year 2020). "
    "Same file as FRASER_LEGAL: fraser_clean.csv. Coverage: 1990-2023, 165 countries."
)
registry_df.loc[registry_df['source_id'] == 'FRASER_REG', 'access_method'] = 'automated_scrape'
registry_df.loc[registry_df['source_id'] == 'FRASER_REG', 'python_approach'] = 'requests — scrape annual report page for XLSX URL'

# FRASER_LEGAL
registry_df.loc[registry_df['source_id'] == 'FRASER_LEGAL', 'notes'] = (
    "Fraser EFW Area 2 (Legal System and Property Rights). Automated pipeline — scrapes annual report page to find XLSX URL. "
    "Also includes Area 4 (Trade Freedom). "
    "Same file as FRASER_REG: fraser_clean.csv. Coverage: 1990-2023, 165 countries."
)
registry_df.loc[registry_df['source_id'] == 'FRASER_LEGAL', 'access_method'] = 'automated_scrape'
registry_df.loc[registry_df['source_id'] == 'FRASER_LEGAL', 'python_approach'] = 'requests — scrape annual report page for XLSX URL'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'].isin(['FRASER_REG', 'FRASER_LEGAL'])][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
       source_id     access_method                                    python_approach
52    FRASER_REG  automated_scrape  requests — scrape annual report page for XLSX URL
53  FRASER_LEGAL  automated_scrape  requests — scrape annual report page for XLSX URL


In [15]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

# HERITAGE_TR
registry_df.loc[registry_df['source_id'] == 'HERITAGE_TR', 'notes'] = (
    "Deprioritized — Fraser Area 4 (Trade Freedom) covers same concept with superior academic methodology. "
    "Heritage adds ~20 countries beyond Fraser but these are outside target sample of 150-160 countries. "
    "No automated download available — Heritage blocks scraping and has no stable data URL. "
    "See framework_decisions.md for full rationale."
)
registry_df.loc[registry_df['source_id'] == 'HERITAGE_TR', 'access_method'] = 'deprioritized'
registry_df.loc[registry_df['source_id'] == 'HERITAGE_TR', 'python_approach'] = 'not built — see notes'

# HERITAGE_PR
registry_df.loc[registry_df['source_id'] == 'HERITAGE_PR', 'notes'] = (
    "Deprioritized — Fraser Area 2 (Legal System) + WJP + V-Dem cover property rights more comprehensively. "
    "Heritage adds nothing not already captured by existing pipelines. "
    "No automated download available — Heritage blocks scraping and has no stable data URL. "
    "See framework_decisions.md for full rationale."
)
registry_df.loc[registry_df['source_id'] == 'HERITAGE_PR', 'access_method'] = 'deprioritized'
registry_df.loc[registry_df['source_id'] == 'HERITAGE_PR', 'python_approach'] = 'not built — see notes'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'].isin(['HERITAGE_TR', 'HERITAGE_PR'])][['source_id', 'access_method']].to_string())

Registry updated
      source_id  access_method
32  HERITAGE_TR  deprioritized
33  HERITAGE_PR  deprioritized


In [16]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

QOG_SOURCES = {
    'KOF_TRADE':     ('via_qog', 'QoG Standard TS — dr_eg (Economic Globalisation). MISMATCH: master PDF calls for trade sub-index specifically.'),
    'PTS':           ('via_qog', 'QoG Standard TS — gd_ptsa (Amnesty), gd_ptsh (HRW), gd_ptss (State Dept).'),
    'OBS':           ('via_qog', 'QoG Standard TS — ibp_obi (Open Budget Index). Biennial, ~120 countries.'),
    'ND_GAIN':       ('via_qog', 'QoG Standard TS — gain_gov (governance readiness), gain_read (readiness). Sub-scores per master PDF.'),
    'BCI':           ('via_qog', 'QoG Standard TS — bci_bci.'),
    'HANSON_SIGMAN': ('via_qog', 'QoG Standard TS — lld_capacity. Double-counting caveat: incorporates V-Dem and other sources we use.'),
    'CCP':           ('via_qog', 'QoG Standard TS — ccp_syst, ccp_market, ccp_civil, ccp_infoacc, ccp_equal. Judicial independence and separation of powers sub-dimensions not clearly captured in QoG CCP subset — gap flagged.'),
    'PEI':           ('via_qog', 'QoG Standard TS — pei_peii_1. Per-election cadence.'),
    'GPI':           ('via_qog', 'QoG Standard TS — gpi_gpi. Optional cross-check per framework decisions.'),
}

for source_id, (access_method, notes) in QOG_SOURCES.items():
    registry_df.loc[registry_df['source_id'] == source_id, 'access_method'] = access_method
    registry_df.loc[registry_df['source_id'] == source_id, 'python_approach'] = 'QoG Standard TS CSV — see 14_qog_pipeline.ipynb'
    registry_df.loc[registry_df['source_id'] == source_id, 'notes'] = notes

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'].isin(QOG_SOURCES.keys())][['source_id', 'access_method']].to_string())

Registry updated
        source_id access_method
14            PTS       via_qog
23            PEI       via_qog
24            CCP       via_qog
26            GPI       via_qog
29            OBS       via_qog
36      KOF_TRADE       via_qog
40        ND_GAIN       via_qog
44  HANSON_SIGMAN       via_qog
58            BCI       via_qog


In [17]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))
registry_df.loc[registry_df['source_id'] == 'WB_INFORMAL', 'access_method'] = 'via_qog'
registry_df.loc[registry_df['source_id'] == 'WB_INFORMAL', 'python_approach'] = 'QoG Standard TS — see 14_qog_pipeline.ipynb'
registry_df.loc[registry_df['source_id'] == 'WB_INFORMAL', 'notes'] = (
    "Informal economy size (% GDP). QoG Standard TS variables: ied_mimic (MIMIC model), ied_dge (DGE model). "
    "Coverage: 1990-2020. Proxy for state administrative reach per State capacity concept."
)
registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'WB_INFORMAL'][['source_id', 'access_method']].to_string())

Registry updated
      source_id access_method
51  WB_INFORMAL       via_qog


In [18]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))
registry_df.loc[registry_df['source_id'] == 'ROMELLI_CBI', 'access_method'] = 'via_qog'
registry_df.loc[registry_df['source_id'] == 'ROMELLI_CBI', 'python_approach'] = 'QoG Standard TS — see 14_qog_pipeline.ipynb'
registry_df.loc[registry_df['source_id'] == 'ROMELLI_CBI', 'notes'] = (
    "CBIE index (Romelli 2022, 2024). Available in QoG Standard TS under cbie_* prefix. "
    "Variables used: cbie_index (overall), cbie_policy, cbie_lending. Coverage: 1923-2023, 155 countries."
)
registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'ROMELLI_CBI'][['source_id', 'access_method']].to_string())

Registry updated
      source_id access_method
10  ROMELLI_CBI       via_qog


In [19]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))
registry_df.loc[registry_df['source_id'] == 'BASEL_AML', 'notes'] = (
    "Primary tier 1 for Financial sector concept. AML/CFT risk composite, 205 countries, annual. "
    "Expert Edition (free for research) gives CSV with 17 indicators. "
    "Public Edition PDF only — no automated download available. "
    "Status: deferred pending Expert Edition access eligibility. "
    "If Expert Edition approved: automate via direct CSV download. "
    "If not: build FATF scraper as alternative (Category 3). "
    "See framework_decisions.md."
)
registry_df.loc[registry_df['source_id'] == 'BASEL_AML', 'access_method'] = 'deferred'
registry_df.loc[registry_df['source_id'] == 'BASEL_AML', 'python_approach'] = 'pending — see notes'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'BASEL_AML'][['source_id', 'access_method']].to_string())

Registry updated
    source_id access_method
31  BASEL_AML      deferred


In [20]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'POWELL_THYNE', 'access_method'] = 'automated_direct'
registry_df.loc[registry_df['source_id'] == 'POWELL_THYNE', 'python_approach'] = 'requests — direct TXT download'
registry_df.loc[registry_df['source_id'] == 'POWELL_THYNE', 'notes'] = (
    "Direct TXT download from Clayton Thyne website. Continuously updated. "
    "Variables: pt_coup_successful, pt_coup_failed, pt_coup_alleged, pt_autocoup. "
    "Version auto-detected from data (V{YYYY}.{MM}.{DD}). Coverage: 1950-2025, 204 countries."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'POWELL_THYNE'][['source_id', 'access_method']].to_string())

Registry updated
       source_id     access_method
15  POWELL_THYNE  automated_direct


In [21]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'UNODC_HOMICIDE', 'access_method'] = 'automated_owid'
registry_df.loc[registry_df['source_id'] == 'UNODC_HOMICIDE', 'python_approach'] = 'requests — OWID CSV (same pattern as TI CPI)'
registry_df.loc[registry_df['source_id'] == 'UNODC_HOMICIDE', 'notes'] = (
    "Homicide rate per 100,000 population. Downloaded via OWID historical panel. "
    "URL: ourworldindata.org/grapher/homicide-rate-unodc.csv. "
    "Coverage: 1990-2024, 208 countries. Annual."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'UNODC_HOMICIDE'][['source_id', 'access_method']].to_string())

Registry updated
         source_id   access_method
13  UNODC_HOMICIDE  automated_owid


In [22]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'IRENA_CAPACITY', 'access_method'] = 'automated_owid'
registry_df.loc[registry_df['source_id'] == 'IRENA_CAPACITY', 'python_approach'] = 'requests — OWID CSV'
registry_df.loc[registry_df['source_id'] == 'IRENA_CAPACITY', 'notes'] = (
    "Share of electricity from renewables (%). Downloaded via OWID (original: IRENA/Ember). "
    "Note: master PDF calls for IRENA Renewables Capacity Statistics (MW) — using renewable share % as proxy. "
    "More interpretable cross-country than raw MW. Coverage: 1990-2025, 226 countries."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'IRENA_CAPACITY'][['source_id', 'access_method']].to_string())

Registry updated
         source_id   access_method
41  IRENA_CAPACITY  automated_owid


In [23]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'IMF_IMAPP', 'access_method'] = 'automated_direct'
registry_df.loc[registry_df['source_id'] == 'IMF_IMAPP', 'python_approach'] = 'requests — direct ZIP download, date auto-detection'
registry_df.loc[registry_df['source_id'] == 'IMF_IMAPP', 'notes'] = (
    "Direct ZIP download from elibrary-areaer.imf.org. "
    "Auto-detects latest vintage by iterating dates backwards from today. "
    "URL pattern: elibrary-areaer.imf.org/Macroprudential/Documents/iMaPP_database-{YYYY}-{MM}-{DD}.zip. "
    "Aggregated from monthly to annual. Variables: tightening actions, loosening actions, net tightening, LTV average. "
    "Coverage: 1990-2024, 135 countries."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'IMF_IMAPP'][['source_id', 'access_method']].to_string())

Registry updated
   source_id     access_method
8  IMF_IMAPP  automated_direct


In [24]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'YALE_EPI', 'access_method'] = 'automated_scrape'
registry_df.loc[registry_df['source_id'] == 'YALE_EPI', 'python_approach'] = 'requests — scrape downloads page for latest year CSV'
registry_df.loc[registry_df['source_id'] == 'YALE_EPI', 'notes'] = (
    "Scrapes epi.yale.edu/downloads for latest year results CSV. "
    "Cross-sectional only — no historical editions in CSV format. "
    "Kept sub-indices: EPI, CCH, ECO, HLT, BDH, MKP, AGR, FSH, WRS, MPE, MHP. "
    "Master PDF: use policy/institutional sub-components selectively — metric pass pending. "
    "Coverage: 180 countries, biennial (even years)."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'YALE_EPI'][['source_id', 'access_method']].to_string())

Registry updated
   source_id     access_method
38  YALE_EPI  automated_scrape


In [27]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'YALE_EPI', 'access_method'] = 'automated_scrape'
registry_df.loc[registry_df['source_id'] == 'YALE_EPI', 'python_approach'] = 'requests — scrape all epi-downloads pages, auto-detect all editions, stack'
registry_df.loc[registry_df['source_id'] == 'YALE_EPI', 'notes'] = (
    "Scrapes all pages of epi.yale.edu/epi-downloads to find all available edition results CSVs. "
    "Years parsed from filenames — no hardcoding. All editions stacked automatically. "
    "Consistent sub-indices across editions: EPI, CCH, ECO, HLT, BDH, AGR, FSH, WRS. "
    "Extended sub-indices (newer editions only): MKP, MPE, MHP. "
    "Methodology differs between editions — limited comparability. "
    "Master PDF: use policy/institutional sub-components selectively — metric pass pending. "
    "Coverage: ~180 countries, biennial."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'YALE_EPI'][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
   source_id     access_method                                                             python_approach
38  YALE_EPI  automated_scrape  requests — scrape all epi-downloads pages, auto-detect all editions, stack


In [28]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'WB_CARBON', 'access_method'] = 'automated_owid'
registry_df.loc[registry_df['source_id'] == 'WB_CARBON', 'python_approach'] = 'requests — OWID CSV'
registry_df.loc[registry_df['source_id'] == 'WB_CARBON', 'notes'] = (
    "Carbon tax instrument coverage per country-year via OWID. "
    "Original source: World Carbon Pricing Database (Dolphin & Merkle). "
    "Three values: No carbon tax / Has a carbon tax / Has a carbon tax only at sub-national level. "
    "Coverage: 1989-2025, 201 countries, annual."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'WB_CARBON'][['source_id', 'access_method']].to_string())

Registry updated
    source_id   access_method
43  WB_CARBON  automated_owid


In [29]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'DPI', 'access_method'] = 'automated_api'
registry_df.loc[registry_df['source_id'] == 'DPI', 'python_approach'] = 'requests — IDB CKAN API, auto-detects latest version'
registry_df.loc[registry_df['source_id'] == 'DPI', 'notes'] = (
    "IDB CKAN API: data.iadb.org/api/3/action/package_show?id=the-database-of-political-institutions-dpi-{year}. "
    "Auto-detects latest version by iterating years backwards. "
    "31 variables: government composition, party fragmentation, checks/balances, electoral system. "
    "Coverage: 1975-2023, 182 countries."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'DPI'][['source_id', 'access_method']].to_string())

Registry updated
   source_id  access_method
25       DPI  automated_api


In [30]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'CIVICUS', 'access_method'] = 'automated_api'
registry_df.loc[registry_df['source_id'] == 'CIVICUS', 'python_approach'] = 'requests — CIVICUS Monitor REST API'
registry_df.loc[registry_df['source_id'] == 'CIVICUS', 'notes'] = (
    "REST API: monitor.civicus.org/api/countries/. No registration required. "
    "Returns all countries with full ratings history. "
    "Latest rating per calendar year retained from multiple interim updates. "
    "5-point categorical: Open/Narrowed/Obstructed/Repressed/Closed. "
    "API only returns data from 2022 — historical data pre-2022 not accessible via API. "
    "Coverage: 199 countries, 2022-present, annual."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'CIVICUS'][['source_id', 'access_method']].to_string())

Registry updated
   source_id  access_method
21   CIVICUS  automated_api


In [31]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

for source_id, notes in {
    'POLITY5': (
        "Sourced via QoG Standard TS dataset. Variables: p_polity2, p_durable. "
        "Polity project not updated since ~2018; QoG version as current as source. "
        "Coverage: 1946-2020, ~167 countries."
    ),
    'NELDA': (
        "Sourced via QoG Standard TS dataset. 10 variables covering election occurrence, "
        "competitiveness, fairness, opposition access. Per-election cadence. "
        "NELDA latest release is 2020; QoG version as current as source. Coverage: ~2000-2020."
    ),
}.items():
    registry_df.loc[registry_df['source_id'] == source_id, 'access_method'] = 'via_qog'
    registry_df.loc[registry_df['source_id'] == source_id, 'python_approach'] = 'QoG Standard TS — see 14_qog_pipeline.ipynb'
    registry_df.loc[registry_df['source_id'] == source_id, 'notes'] = notes

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'].isin(['POLITY5','NELDA'])][['source_id','access_method']].to_string())

Registry updated
   source_id access_method
56     NELDA       via_qog
65   POLITY5       via_qog


In [2]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'IDEA_PARTIP', 'access_method'] = 'manual_download'
registry_df.loc[registry_df['source_id'] == 'IDEA_PARTIP', 'python_approach'] = 'pandas — auto-detects latest gsod_indices_v{N}.csv in Downloads'
registry_df.loc[registry_df['source_id'] == 'IDEA_PARTIP', 'notes'] = (
    "Manual CSV download from idea.int/democracytracker/gsod-indices. "
    "Pipeline auto-detects latest version (gsod_indices_v{N}.csv) in Downloads. "
    "13 estimates: participation, civil society, civic engagement, electoral participation, "
    "representation, credible elections, judicial independence, etc. "
    "Coverage: 1990-2025, 174 countries. Version auto-detected from filename."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'IDEA_PARTIP'][['source_id', 'access_method']].to_string())

Registry updated
      source_id    access_method
57  IDEA_PARTIP  manual_download


In [3]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'PEW_GRI', 'access_method'] = 'manual_download'
registry_df.loc[registry_df['source_id'] == 'PEW_GRI', 'python_approach'] = 'pandas — auto-detects Pew GRI ZIP in Downloads, extracts CSV'
registry_df.loc[registry_df['source_id'] == 'PEW_GRI', 'notes'] = (
    "Manual ZIP download from pewresearch.org (free account required). "
    "Pipeline auto-detects ZIP (Global-Restrictions-on-Religion*.zip) and extracts CSV. "
    "Headline indices: GRI (Government Restrictions Index, 0-10), SHI (Social Hostilities Index, 0-10). "
    "79 question-level columns dropped. Coverage: 198 countries, annual."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'PEW_GRI'][['source_id', 'access_method']].to_string())

Registry updated
   source_id    access_method
54   PEW_GRI  manual_download


In [4]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'IMF_FISCAL_RULES', 'access_method'] = 'manual_download'
registry_df.loc[registry_df['source_id'] == 'IMF_FISCAL_RULES', 'python_approach'] = 'pandas — auto-detects Fiscal Rules Excel in Downloads; columns by position'
registry_df.loc[registry_df['source_id'] == 'IMF_FISCAL_RULES', 'notes'] = (
    "Manual Excel download from imf.org. Pipeline auto-detects file (*Fiscal Rules*.xlsx) in Downloads. "
    "Rules sheet has 4-row nested header — columns selected by position. "
    "FRAGILITY: if IMF restructures the template column order, the position mapping needs review. "
    "Four binary rule-type indicators (expenditure, revenue, budget balance, debt) plus count. "
    "Coverage: 123 countries."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'IMF_FISCAL_RULES'][['source_id', 'access_method']].to_string())

Registry updated
          source_id    access_method
6  IMF_FISCAL_RULES  manual_download


In [5]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'IMF_FISCAL_RULES', 'access_method'] = 'manual_download'
registry_df.loc[registry_df['source_id'] == 'IMF_FISCAL_RULES', 'python_approach'] = 'pandas — auto-detects Excel in Downloads; robust name-based column selection from nested header'
registry_df.loc[registry_df['source_id'] == 'IMF_FISCAL_RULES', 'notes'] = (
    "Manual Excel download from imf.org. Pipeline auto-detects file (*Fiscal Rules*.xlsx) in Downloads. "
    "Rules sheet has 4-row nested header — columns selected by NAME (composite key from forward-filled "
    "parent rows + leaf row), not position. Survives column reordering; fails loudly if a section is renamed. "
    "Measures presence AND quality per rule type: legal basis (1-5 ordinal), enforcement, compliance, "
    "plus independent monitoring, correction mechanism. Coverage: 123 countries."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'IMF_FISCAL_RULES'][['source_id', 'access_method']].to_string())

Registry updated
          source_id    access_method
6  IMF_FISCAL_RULES  manual_download


In [6]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'IMF_IMAPP', 'access_method'] = 'automated_direct'
registry_df.loc[registry_df['source_id'] == 'IMF_IMAPP', 'python_approach'] = 'requests — direct ZIP, date auto-detection; breadth from MaPP sheet'
registry_df.loc[registry_df['source_id'] == 'IMF_IMAPP', 'notes'] = (
    "Direct ZIP from elibrary-areaer.imf.org, latest vintage auto-detected by date iteration. "
    "Cumulative toolkit BREADTH: count of distinct instruments (16; 'Other' excluded) ever activated "
    "up to each year, total + by category (borrower-based, capital-based, liquidity-funding, provision-reserve-tax). "
    "Direction excluded. Proxy for framework development — NOT in-force, NOT quality directly (see FSAP). "
    "RR noisy per IMF. Coverage: 1990-2024, 135 countries."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'IMF_IMAPP'][['source_id', 'access_method']].to_string())

Registry updated
   source_id     access_method
8  IMF_IMAPP  automated_direct


In [7]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'WB_CARBON', 'access_method'] = 'automated_direct'
registry_df.loc[registry_df['source_id'] == 'WB_CARBON', 'python_approach'] = 'requests — month-stamped xlsx, date auto-detection; Gen Info spine + Unique ID joins'
registry_df.loc[registry_df['source_id'] == 'WB_CARBON', 'notes'] = (
    "WB Carbon Pricing Dashboard month-stamped xlsx (data_{MM}_{YYYY}.xlsx), latest auto-detected, "
    "retry/backoff for rate limits. Measures: existence, price (US$/tCO2e), revenue (US$m), "
    "jurisdictional coverage % (current snapshot). National-only (explicit ISO3 dict; fail-safe — "
    "unmapped jurisdictions excluded). EU ETS expanded to members for intensive measures "
    "(price/coverage/existence) but NOT revenue (bloc total). Within-country coverage = max. "
    "Absence = INFERRED not verified non-existence. ⚠️ ~71 countries (carbon pricing concentrated). "
    "Replaces prior OWID binary. Revenue/GDP at metric pass. MANUAL UPDATE: national dict + EU member list."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'WB_CARBON'][['source_id', 'access_method']].to_string())

Registry updated
    source_id     access_method
43  WB_CARBON  automated_direct


In [8]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'CLIMATE_LAWS', 'access_method'] = 'manual_download'
registry_df.loc[registry_df['source_id'] == 'CLIMATE_LAWS', 'python_approach'] = 'pandas — auto-detects Document_Data_Download*.csv in Downloads'
registry_df.loc[registry_df['source_id'] == 'CLIMATE_LAWS', 'notes'] = (
    "Manual CSV from climate-laws.org (free registration form). Pipeline auto-detects "
    "Document_Data_Download*.csv in Downloads. Cumulative stock of domestic climate laws/policies "
    "per country-year + new_laws flow. UNFCCC excluded; Legislative+Executive kept; deduped to Family ID. "
    "NATIONAL-ONLY (EU-level EUR docs dropped to avoid double-count with national transpositions; "
    "subnational dropped). Coverage: 199 countries."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'CLIMATE_LAWS'][['source_id', 'access_method']].to_string())

Registry updated
       source_id    access_method
39  CLIMATE_LAWS  manual_download


In [9]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'ODIN', 'access_method'] = 'manual_download'
registry_df.loc[registry_df['source_id'] == 'ODIN', 'python_approach'] = 'pandas — auto-detects ODIN ZIP in Downloads (content-validated), stacks per-edition Excels'
registry_df.loc[registry_df['source_id'] == 'ODIN', 'notes'] = (
    "Manual ZIP from odin.opendatawatch.com/data. Pipeline globs *data*.zip and validates by contents "
    "(ZIP must contain year-named Excels). Sub-scores are TRANSPARENT SIMPLE-MEAN aggregation of ODIN's "
    "per-category element scores — NOT ODIN's official 0-100 index. Raw ~0-2 scale (ranking valid, "
    "downstream-normalized). Biennial editions stacked. Overlaps substantially with IMF SPI. "
    "Coverage: 200 countries. MANUAL: keep only the current ODIN ZIP in Downloads to avoid ambiguous match."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'ODIN'][['source_id', 'access_method']].to_string())

Registry updated
   source_id    access_method
27      ODIN  manual_download


In [10]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'IRENA_POLICY', 'access_method'] = 'deprioritized'
registry_df.loc[registry_df['source_id'] == 'IRENA_POLICY', 'python_approach'] = 'none — not built'
registry_df.loc[registry_df['source_id'] == 'IRENA_POLICY', 'notes'] = (
    "DEPRIORITIZED — not built. No clean downloadable renewable-policy dataset exists. "
    "The IRENA Stats Tool .xlsb is renewable STATISTICS (capacity/generation/finance), not policy — "
    "duplicates the IRENA capacity pipeline. IRENA's renewable-policy work is report-based analysis, not a "
    "structured panel. The only structured renewable-policy data is the joint IEA/IRENA Policies & Measures "
    "DB (api.iea.org/policies?csv=true), which has no clean renewable-energy filter in its current taxonomy "
    "and would broadly duplicate Climate Laws. DEFERRED CANDIDATE: IRENA national renewable-energy TARGETS "
    "(a genuine ambition signal) via the planned PDF-extraction infrastructure — not available as a clean download."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated — IRENA_POLICY deprioritized")
print(registry_df[registry_df['source_id'] == 'IRENA_POLICY'][['source_id', 'access_method']].to_string())

Registry updated — IRENA_POLICY deprioritized
       source_id  access_method
42  IRENA_POLICY  deprioritized


In [12]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'TI_POLFINANCE', 'access_method'] = 'automated_direct'
registry_df.loc[registry_df['source_id'] == 'TI_POLFINANCE', 'python_approach'] = 'requests — IDEA export .xlsx endpoint (themeId=302)'
registry_df.loc[registry_df['source_id'] == 'TI_POLFINANCE', 'notes'] = (
    "Source is International IDEA Political Finance Database (NOT Transparency International). Automated "
    ".xlsx export: idea.int/data-tools/export?type=region_only&themeId=302&world=all. Score "
    "polfin_transparency_integrity (0-1) = equal-weighted mean of 20 directionally-defensible binary "
    "questions (disclosure/reporting + anti-corruption source bans + state-resource & vote-buying bans). "
    "Contested questions (contribution/spending limits, public funding, corporate/union bans) excluded — "
    "directionality rationale in framework_decisions.md. DE JURE only (rules on paper, not enforcement). "
    "Reliability floor <10/20 answered -> NaN. Wave-updated cross-section. 180 countries. "
    "MANUAL: the 20-question inclusion list encodes a directionality judgment — revisit if IDEA revises questions."
)

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'TI_POLFINANCE'][['source_id', 'access_method']].to_string())

Registry updated
        source_id     access_method
48  TI_POLFINANCE  automated_direct


In [13]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

gdb_notes = (
    "DEPRIORITIZED — not built. Global Data Barometer (successor to defunct Open Data Barometer). "
    "Accessible (per-module CSVs at storage.googleapis.com/gdb-files), but thin (~43-109 countries, "
    "edition-unstable, not a panel) and duplicates ODIN's open-data coverage already in the framework. "
    "Does NOT fill Concept 25's actual measurement gaps (procurement transparency, lobbying transparency, "
    "de facto vs de jure disclosure). Marginal contribution over ODIN judged too small to justify a pipeline."
)

if (registry_df['source_id'] == 'GLOBAL_DATA_BAROMETER').any():
    registry_df.loc[registry_df['source_id'] == 'GLOBAL_DATA_BAROMETER', 'access_method'] = 'deprioritized'
    registry_df.loc[registry_df['source_id'] == 'GLOBAL_DATA_BAROMETER', 'python_approach'] = 'none — not built'
    registry_df.loc[registry_df['source_id'] == 'GLOBAL_DATA_BAROMETER', 'notes'] = gdb_notes
    print("Updated existing GDB registry row")
else:
    new_row = {'source_id': 'GLOBAL_DATA_BAROMETER', 'access_method': 'deprioritized',
               'python_approach': 'none — not built', 'notes': gdb_notes}
    registry_df = pd.concat([registry_df, pd.DataFrame([new_row])], ignore_index=True)
    print("Added new GDB registry row (deprioritized)")

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print(registry_df[registry_df['source_id'] == 'GLOBAL_DATA_BAROMETER'][['source_id', 'access_method']].to_string())

Updated existing GDB registry row
                source_id  access_method
59  GLOBAL_DATA_BAROMETER  deprioritized


In [14]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

rti_notes = (
    "Global RTI Rating (CLD / Access Info Europe). PRIMARY tier-1: Govt transparency (C25) + Media "
    "Freedom (C23). Automated — full scores table parsed from country-data page HTML via pandas.read_html; "
    "deficit-list (no-law countries) xlsx URL extracted dynamically from the page (no hardcoded date). "
    "rti_total (0-150, CLD's own weighted sum of 61 indicators) + 7 category sub-scores + rti_law_year. "
    "DE JURE only (not implementation). 142 rated (has_rti_law=1); 54 no-law (has_rti_law=0) floored at "
    "min-1SD clamped >=0. 196 countries. Cross-section; history deferred. MANUAL: add ISO3 mappings if "
    "new countries appear unmapped (pipeline prints unmapped names)."
)

if (registry_df['source_id'] == 'RTI_RATING').any():
    registry_df.loc[registry_df['source_id'] == 'RTI_RATING', 'access_method'] = 'automated_direct'
    registry_df.loc[registry_df['source_id'] == 'RTI_RATING', 'python_approach'] = 'requests + pandas.read_html — parse scores table from page HTML; dynamic deficit-file URL'
    registry_df.loc[registry_df['source_id'] == 'RTI_RATING', 'notes'] = rti_notes
    print("Updated RTI_RATING registry row")
else:
    new_row = {'source_id': 'RTI_RATING', 'access_method': 'automated_direct',
               'python_approach': 'requests + pandas.read_html — parse scores table from page HTML; dynamic deficit-file URL',
               'notes': rti_notes}
    registry_df = pd.concat([registry_df, pd.DataFrame([new_row])], ignore_index=True)
    print("Added RTI_RATING registry row")

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print(registry_df[registry_df['source_id'] == 'RTI_RATING'][['source_id', 'access_method']].to_string())

Updated RTI_RATING registry row
     source_id     access_method
47  RTI_RATING  automated_direct


In [15]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

areaer_notes = (
    "IMF AREAER — FARI capital-account restrictiveness index (de jure). PRIMARY tier-1. MANUAL: portal "
    "WAF-blocked to code + JS-gated; export FARI Indices by hand (Indices tab → FARI Aggregate + FARI-FDI "
    "families, all countries, annual, full range) to Downloads; pipeline auto-detects latest "
    "FARIReportByCountry*.xlsx. fari_aggregate + fari_fdi_aggregate (0-1, higher=more restrictive) primary; "
    "inflow/outflow supplementary. 194 countries 1999-2024 (2024 partial). Complemented by Chinn-Ito "
    "(automated) + Reinhart-Rogoff (de facto regime). ACI downloaded, not built (deferred). IFS-code file, "
    "ISO3 mapped via name + MANUAL_ISO3 dict (needs pycountry)."
)

if (registry_df['source_id'] == 'IMF_AREAER').any():
    registry_df.loc[registry_df['source_id'] == 'IMF_AREAER', 'access_method'] = 'manual_download'
    registry_df.loc[registry_df['source_id'] == 'IMF_AREAER', 'python_approach'] = 'manual export to Downloads (portal WAF-blocked); auto-detect latest FARIReportByCountry*.xlsx, reshape wide->long, ISO3 via name'
    registry_df.loc[registry_df['source_id'] == 'IMF_AREAER', 'notes'] = areaer_notes
    print("Updated IMF_AREAER registry row")
else:
    new_row = {'source_id': 'IMF_AREAER', 'access_method': 'manual_download',
               'python_approach': 'manual export to Downloads (portal WAF-blocked); auto-detect latest FARIReportByCountry*.xlsx, reshape wide->long, ISO3 via name',
               'notes': areaer_notes}
    registry_df = pd.concat([registry_df, pd.DataFrame([new_row])], ignore_index=True)
    print("Added IMF_AREAER registry row")

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print(registry_df[registry_df['source_id'] == 'IMF_AREAER'][['source_id', 'access_method']].to_string())

Updated IMF_AREAER registry row
    source_id    access_method
7  IMF_AREAER  manual_download


In [17]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

ci_approach = ("requests — scrape faculty page for newest kaopen_YYYY.xls (year parsed "
               "dynamically), download to RAW_DIR, pandas.read_excel(engine='xlrd'), "
               "ISO3 native + ZAR->COD remap")
ci_notes = (
    "Chinn-Ito KAOPEN capital-account openness (de jure). PRIMARY tier-1 — Macro policy framework "
    "quality (Cat 5), one of six primaries; derivative / cross-check to AREAER FARI (nb 32), NOT more "
    "authoritative. AUTOMATED (31_chinn_ito_pipeline): faculty page (web.pdx.edu/~ito) scraped for "
    "newest kaopen_YYYY.xls, year parsed dynamically (no hardcode); fragile personal-page URL "
    "(fallback in instructions_data_maintenance.md). kaopen (raw PCA, higher=MORE open; OPPOSITE sign "
    "to FARI) primary; kaopen_norm (0-1) supplementary. Ships ISO3; ZAR->COD remap; ANT (dead code) "
    "retained; Serbia/Timor unscored -> dropped. 181 valid-ISO3 + ANT = 182, 1970-2023. VERSION "
    "NON-STABLE: PCA recomputed each release -> full-replace, never append. Needs xlrd engine for .xls."
)

if (registry_df['source_id'] == 'CHINN_ITO').any():
    registry_df.loc[registry_df['source_id'] == 'CHINN_ITO', 'access_method'] = 'automated_scrape'
    registry_df.loc[registry_df['source_id'] == 'CHINN_ITO', 'python_approach'] = ci_approach
    registry_df.loc[registry_df['source_id'] == 'CHINN_ITO', 'notes'] = ci_notes
    print("Updated CHINN_ITO registry row")
else:
    registry_df = pd.concat([registry_df, pd.DataFrame([{
        'source_id': 'CHINN_ITO',
        'source_name': 'Chinn-Ito Index (KAOPEN) — Capital Account Openness',
        'access_method': 'automated_scrape',
        'python_approach': ci_approach,
        'update_frequency': 'annual',
        'coverage_countries': 182,
        'coverage_years': '1970-present',
        'highest_tier': 'P1',
        'notes': ci_notes,
    }])], ignore_index=True)
    print("Added CHINN_ITO registry row")

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print(registry_df[registry_df['source_id'] == 'CHINN_ITO']
      [['source_id', 'access_method', 'highest_tier', 'coverage_countries']].to_string(index=False))

Added CHINN_ITO registry row
source_id    access_method highest_tier  coverage_countries
CHINN_ITO automated_scrape           P1                 182


In [19]:
# ── 02 · Register PEFA (upsert) ──────────────────────────────────────────────
import os, pandas as pd
REG = os.path.join(PROCESSED_DIR, "source_registry.csv")
reg = pd.read_csv(REG)

row = {
    "source_id": "PEFA",
    "source_name": "PEFA — Public Expenditure and Financial Accountability (PFM quality)",
    "access_method": "manual_download",
    "python_approach": ("33_pefa_pipeline: glob assessments_*.csv from Downloads -> snapshot RAW_DIR; "
                        "filter Framework==PEFA_FRAMEWORK & national; dedup latest/country; melt wide->long; "
                        "A-D->numeric (7pt indicator / 4pt dimension, '*' stripped+flagged, NU/NR/blank dropped); "
                        "pycountry ISO3 + OVERRIDES"),
    "update_frequency": "rolling",
    "coverage_countries": 85,
    "coverage_years": "2017-present",
    "highest_tier": "P1",
    "notes": ("PFM quality (BUILT, 33_pefa_pipeline). Feeds fiscal-governance concepts — candidates: budget "
              "transparency (PI-9), fiscal-risk reporting (PI-10), audit & legislative scrutiny (PI-26/30/31), "
              "debt (PI-13), procurement (PI-24); concept mapping downstream. Source = PEFA 'Scores Downloads' CSV "
              "(Framework=PEFA_FRAMEWORK, Type=National, Status=Final), manual download -> Downloads -> RAW_DIR "
              "snapshot; filter framework+national, dedup latest/country, A-D->numeric (7pt indicator/4pt dim), "
              "pycountry ISO3. 2016 core = 85 countries / 31 indicators / 2017-2026. 2011 backfill DEFERRED "
              "(stale: Brazil/India/Norway 16-18 yrs) — see framework_decisions.md. data_as_of derived from "
              "max assessment_year; manual knob PEFA_FRAMEWORK documented in instructions_data_maintenance.md."),
}

if (reg['source_id'] == 'PEFA').any():                      # upsert
    for k, v in row.items():
        reg.loc[reg['source_id'] == 'PEFA', k] = v
    action = "updated"
else:
    reg = pd.concat([reg, pd.DataFrame([row])], ignore_index=True)
    action = "appended"

reg.to_csv(REG, index=False)
print(f"PEFA {action}. registry rows: {len(reg)}")
print(reg[reg['source_id'] == 'PEFA'].to_string(index=False))

PEFA updated. registry rows: 69
source_id                                                          source_name   access_method                                                                                                                                                                                                                                                                      python_approach update_frequency  coverage_countries coverage_years highest_tier                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [11]:
# ============================================================
# CELL N — Add/update OECD_TFI in source registry
# Derives all dynamic values from the saved tfi_clean.csv and
# download_log — no dependency on the TFI pipeline kernel.
# ============================================================
import pandas as pd
import os

registry_path = os.path.join(PROCESSED_DIR, "source_registry.csv")
registry = pd.read_csv(registry_path)

# --- Derive values from saved pipeline outputs ---
# Read tfi_clean.csv to get actual coverage stats.
tfi = pd.read_csv(os.path.join(PROCESSED_DIR, "tfi_clean.csv"))
n_countries = tfi["iso3"].nunique()
years_present = sorted(tfi["year"].unique())
coverage_years = "/".join(str(y) for y in years_present)
data_as_of = str(max(years_present))   # latest year in the file

print(f"Derived from tfi_clean.csv: {n_countries} countries, years {coverage_years}, as-of {data_as_of}")

new_row = {
    "source_id":           "OECD_TFI",
    "source_name":         "OECD Trade Facilitation Indicators",
    "category":            "Trade governance",
    "tier":                "P1",
    "highest_tier":        "P1",
    "access_method":       "manual_download",
    "python_approach":     "pd.read_excel",
    "update_frequency":    "biennial",
    "data_as_of_date":     data_as_of,
    "coverage_countries":  str(n_countries),
    "coverage_years":      coverage_years,
    "coverage":            f"{n_countries} countries, {coverage_years}",
    "notes":               "Composite average of 11 sub-indicators (A-K). CYC Overview table download. Sub-indicators and 2024 edition deferred to Cat-1 PDF batch."
}

# Update if exists, append if new.
# Cast all target columns to object dtype first to avoid dtype conflicts
# (e.g. data_as_of_date stored as float64 in existing rows).
if "OECD_TFI" in registry["source_id"].values:
    for col, val in new_row.items():
        if col in registry.columns:
            registry[col] = registry[col].astype(object)
            registry.loc[registry["source_id"] == "OECD_TFI", col] = val
    print("Updated existing OECD_TFI row")
else:
    registry = pd.concat([registry, pd.DataFrame([new_row])], ignore_index=True)
    print("Appended new OECD_TFI row")
    
registry.to_csv(registry_path, index=False)
print(f"Registry saved. Total rows: {len(registry)}")
print(registry[registry["source_id"] == "OECD_TFI"].to_string())

Derived from tfi_clean.csv: 164 countries, years 2017/2019/2022, as-of 2022
Updated existing OECD_TFI row
Registry saved. Total rows: 69
   source_id                         source_name    access_method python_approach update_frequency coverage_countries  coverage_years highest_tier                                                                                                                                    notes          category tier data_as_of_date                       coverage
35  OECD_TFI  OECD Trade Facilitation Indicators  manual_download   pd.read_excel         biennial                164  2017/2019/2022           P1  Composite average of 11 sub-indicators (A-K). CYC Overview table download. Sub-indicators and 2024 edition deferred to Cat-1 PDF batch.  Trade governance   P1            2022  164 countries, 2017/2019/2022


In [12]:
# ============================================================
# NOTEBOOK 02 — NEW FINAL CELL (permanent; not diagnostic)
# Registers IMF_AREAER_ERREGIME (de facto exchange-rate-REGIME classification) in
# source_registry.csv, following the add-if-not-exists pattern used for RTI/GDB/
# IMF_AREAER. This source is DISTINCT from IMF_AREAER (= FARI capital-account index).
# coverage_countries and the vintage are DERIVED from the built clean CSV, so this
# cell needs NO hardcoded counts/dates — re-running after a data refresh auto-updates
# them. DEPENDENCY: run 37_areaer_defacto_er_pipeline.ipynb first (produces the CSV).
# ============================================================
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

# --- Schema guard: fail loudly rather than silently adding stray columns -------
# (I could not read source_registry.csv directly, so this verifies my assumed schema
#  at runtime; if it fires, it prints the ACTUAL columns so the row can be aligned.)
_expected_cols = {"source_id", "source_name", "access_method", "python_approach",
                  "update_frequency", "coverage_countries", "coverage_years",
                  "highest_tier", "notes"}
_unknown = _expected_cols - set(registry_df.columns)
assert not _unknown, f"Registry schema differs — missing {_unknown}. Actual columns: {list(registry_df.columns)}"

# --- Derive coverage + vintage FROM the built clean CSV (no hardcoded numbers) --
_clean_path = os.path.join(PROCESSED_DIR, "areaer_er_clean.csv")
assert os.path.exists(_clean_path), f"Run 37_areaer_defacto_er_pipeline.ipynb first — {_clean_path} not found."
_areaer   = pd.read_csv(_clean_path)
_n_ctry   = int(_areaer["country_code"].nunique())               # derived count (e.g. 195)
_vint_vals = _areaer["areaer_as_of"].dropna().unique()           # single-snapshot expected
_vintage  = _vint_vals[0] if len(_vint_vals) == 1 else "multiple — see data"

# --- Static descriptive note (changes only if METHODOLOGY changes, not per refresh) -
areaer_er_notes = (
    "IMF AREAER De Facto Exchange Rate Arrangement Classification (Annual Report, Appendix II.9). "
    "PRIMARY tier-1 for Macroeconomic policy framework (Concept 8) — the IMF-native de facto ER-regime "
    "source; SUPERSEDES Reinhart-Rogoff de facto regime. DISTINCT from IMF_AREAER (= FARI capital-account "
    "restrictiveness). HAND-TRANSCRIBED to data/raw/areaer_defacto_regime.csv (AREAER Online paywalled; "
    "borderless-matrix PDF has no reliable automated extraction), validated at transcription against the "
    "PDF's per-category (row) AND per-column (MPF) country-count checksums. Pipeline "
    "(37_areaer_defacto_er_pipeline.ipynb) encodes flexibility ordinal 1-10 (matrix order; other_managed=8 "
    "is a RESIDUAL, not a flexibility rank) + IMF 4-way group, plus areaer_mpf (monetary-policy framework "
    "incl. inflation_targeting flag), anchor_currency, and reclassified (YYYY-MM regime-change recency). "
    "Cross-section snapshot; NO year (vintage = areaer_as_of). MANUAL REFRESH: re-transcribe reclassified "
    "countries + bump areaer_as_of in the source CSV, then re-run pipeline + this cell — NO code changes. "
    "See instructions_data_maintenance.md."
)

# --- Assemble the registry row (schema matches the base 'sources' list) --------
areaer_er_row = {
    "source_id":          "IMF_AREAER_ERREGIME",
    "source_name":        "IMF AREAER — De Facto Exchange Rate Arrangement Classification",
    "access_method":      "manual_transcribed",   # NEW value: hand-transcribed from PDF matrix (not a file download)
    "python_approach":    "pandas — reads hand-transcribed data/raw/areaer_defacto_regime.csv; see 37_areaer_defacto_er_pipeline.ipynb",
    "update_frequency":   "annual",
    "coverage_countries": _n_ctry,                 # DERIVED from clean CSV
    "coverage_years":     f"{_vintage} (single-snapshot)",  # DERIVED vintage
    "highest_tier":       "P1",
    "notes":              areaer_er_notes,
}

# --- Add-if-not-exists (idempotent; safe to re-run) ---------------------------
if (registry_df["source_id"] == "IMF_AREAER_ERREGIME").any():
    for k, v in areaer_er_row.items():
        registry_df.loc[registry_df["source_id"] == "IMF_AREAER_ERREGIME", k] = v
    print("Updated existing IMF_AREAER_ERREGIME registry row")
else:
    registry_df = pd.concat([registry_df, pd.DataFrame([areaer_er_row])], ignore_index=True)
    print("Added IMF_AREAER_ERREGIME registry row")

# --- Persist and confirm ------------------------------------------------------
registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print(registry_df[registry_df["source_id"] == "IMF_AREAER_ERREGIME"][
    ["source_id", "access_method", "coverage_countries", "coverage_years", "highest_tier"]].to_string(index=False))

Added IMF_AREAER_ERREGIME registry row
          source_id      access_method  coverage_countries               coverage_years highest_tier
IMF_AREAER_ERREGIME manual_transcribed                 195 2025-04-30 (single-snapshot)           P1


In [14]:
# ============================================================
# NOTEBOOK 02 — NEW FINAL CELL (permanent; not diagnostic)
# Registers WB_BRSS in source_registry.csv (add-if-not-exists, per RTI/AREAER pattern).
# coverage_countries derived from the built clean CSV -> no hardcoded counts.
# DEPENDENCY: run 38_wb_brss_pipeline.ipynb first (produces wb_brss_clean.csv).
# ============================================================
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

_expected = {"source_id","source_name","access_method","python_approach","update_frequency",
             "coverage_countries","coverage_years","highest_tier","notes"}
_missing = _expected - set(registry_df.columns)
assert not _missing, f"Registry schema differs — missing {_missing}. Actual: {list(registry_df.columns)}"

_clean = os.path.join(PROCESSED_DIR, "wb_brss_clean.csv")
assert os.path.exists(_clean), f"Run 38_wb_brss_pipeline.ipynb first — {_clean} not found."
_df = pd.read_csv(_clean)
_n_total   = int(_df["country_code"].nunique())
_n_reliable= int(_df.loc[_df["brss_reliable"]==True, "country_code"].nunique())

wb_brss_notes = (
    "World Bank Bank Regulation & Supervision Survey (5th wave, 2019; reference year 2016). SUPPLEMENTARY — "
    "Concept 9 banking-supervision leg (complements FATF AML/CFT). DE JURE regulatory-STRINGENCY (rules-on-paper), "
    "NOT supervisory effectiveness — advanced economies mid-pack is CORRECT, validated against Anginer et al. (2019) "
    "(reproduces their HI/DEV directional findings, zero construct inversions). Transparent construct-aligned "
    "select-and-score (NOT published BCL indices): 9 sub-constructs, comparable >=80%-coverage items only; "
    "Activity Restrictions dropped (contested directionality); provisioning/macropru trimmed of prescriptive items "
    "penalizing IFRS-9/principle-based regimes. Equal-weight within construct (revisit post-v1); 5 constructs 2x in "
    "overall. brss_reliable = coverage>=70%. Cross-section snapshot, NO year. FROZEN wave (irregular: "
    "2001/03/07/11/19) — auto-alert on 6th wave; RE-VALIDATE INCLUDED_QUESTIONS on any new wave. Auto-discover fetch "
    "(WB catalog); CC-BY-4.0. Pipeline 38_wb_brss_pipeline.ipynb. See instructions_data_maintenance.md."
)
wb_brss_row = {
    "source_id":          "WB_BRSS",
    "source_name":        "World Bank Bank Regulation & Supervision Survey (BRSS)",
    "access_method":      "auto_discover_download",   # scrape catalog -> newest .xlsx -> cache
    "python_approach":    "pandas/openpyxl — reads cached wb_brss_2019.xlsx; see 38_wb_brss_pipeline.ipynb",
    "update_frequency":   "irregular (survey waves ~4-8yr; frozen at 2019)",
    "coverage_countries": _n_total,                   # derived (161; 155 reliable)
    "coverage_years":     "2019 wave / 2016 reference (single-snapshot)",
    "highest_tier":       "Supplementary",
    "notes":              wb_brss_notes,
}

if (registry_df["source_id"] == "WB_BRSS").any():
    for k, v in wb_brss_row.items():
        registry_df.loc[registry_df["source_id"] == "WB_BRSS", k] = v
    print("Updated existing WB_BRSS registry row")
else:
    registry_df = pd.concat([registry_df, pd.DataFrame([wb_brss_row])], ignore_index=True)
    print("Added WB_BRSS registry row")

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print(f"coverage_countries={_n_total} ({_n_reliable} reliable)")
print(registry_df[registry_df["source_id"]=="WB_BRSS"][
    ["source_id","access_method","coverage_countries","highest_tier"]].to_string(index=False))

Added WB_BRSS registry row
coverage_countries=161 (155 reliable)
source_id          access_method  coverage_countries  highest_tier
  WB_BRSS auto_discover_download                 161 Supplementary


In [5]:
# ============================================================
# ASCOR — source_registry entry (TRIMMED). Registry = index; reasoning lives in
# framework_decisions.md, operational refresh detail in download_log.
# ============================================================
import os, pandas as pd
REG_PATH = os.path.join(PROCESSED_DIR, "source_registry.csv")
reg = pd.read_csv(REG_PATH)

ascor_row = {
    "source_id":   "ASCOR",
    "source_name": "ASCOR — Assessing Sovereign Climate-related Opportunities and Risks (TPI Centre, LSE)",
    "access_method": "manual_download",
    "python_approach": "pandas/openpyxl — reads data/raw/ascor_*.xlsx; see 40_ascor_pipeline.ipynb",
    "update_frequency": "annual",
    "coverage_countries": 85,
    "coverage_years": "2023-2025 (3 rounds: 25 / 70 / 85 countries)",
    "highest_tier": "P2",
    "notes": (
        "Concept 12 climate-policy leg. Scored metric = ascor_climate_governance (5 universally-answered "
        "areas, share-of-applicable-Yes, fixed 0-1 anchor, unnormalized). TIER P2 PROVISIONAL — 44% of "
        "the sovereign core, 35 non-high-income; confirm at Step-1. CC BY-NC (non-commercial). v2.0 "
        "overhaul pending. Rationale + exemption structure: framework_decisions.md. Refresh: "
        "instructions_data_maintenance.md."
    ),
}

idx = reg.index[reg["source_id"] == "ASCOR"]
if len(idx):
    for k, v in ascor_row.items(): reg.at[idx[0], k] = v
    action = "UPDATED"
else:
    reg = pd.concat([reg, pd.DataFrame([ascor_row])], ignore_index=True); action = "APPENDED"
reg.to_csv(REG_PATH, index=False)
print(f"{action} ASCOR — notes length: {len(ascor_row['notes'])} chars")

UPDATED ASCOR — notes length: 398 chars
